# 2. Search Evaluation (Retrieval Metrics)

This stage answers: **does search retrieve the right document?**

It needs **no LLM** — it's pure, free, fast. We run it over the full 395-question
ground truth.

The recipe:
1. Take a ground-truth question (we know its correct doc id).
2. Run search, get the top-5 results.
3. Build a *relevance list*: `1` where a result is the correct doc, else `0`.
   e.g. correct doc at rank 1 -> `[1, 0, 0, 0, 0]`.
4. Aggregate all lists into two metrics: **Hit Rate** and **MRR**.

> We read the **freshly generated** `data/ground_truth.csv` (from notebook 1), not
> the shipped `ground_truth-new.csv` — the latter is stale vs the live FAQ (see
> notebook 1), which would deflate the metrics.

In [1]:
import pandas as pd
df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
df_ground_truth.head()

,question,document
0,Am I too late to enroll if I found this class ...,74eb249bbf
1,Started late - is it still possible to sign up?,74eb249bbf
2,Can I hop in even though the course already st...,74eb249bbf
3,If I register now will I still be able to earn...,74eb249bbf
4,"Missed the beginning, can I still get in and g...",74eb249bbf


In [2]:
# Build the minsearch index over the llm-zoomcamp docs.
from ingest import load_faq_data, build_index
documents = [d for d in load_faq_data() if d["course"] == "llm-zoomcamp"]
index = build_index(documents)

A search function: `boost_dict` multiplies the score of a field. Lexical match
on the short **question** is a stronger relevance signal than a match buried in a
long **answer**, so we boost `question`.

In [3]:
def text_search(query):
    return index.search(query, num_results=5, boost_dict={"question": 3.0, "section": 0.5})

In [4]:
# Relevance list for one question.
q = ground_truth[0]
doc_id = q["document"]
results = text_search(q["question"])
[int(d["id"] == doc_id) for d in results]  # [1,0,0,0,0] means: correct doc is at rank 1

[0, 1, 0, 0, 0]

In [5]:
# Build relevance lists for ALL questions (this is the expensive-but-free part).
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])
    return [int(d["id"] == doc_id) for d in results]

def compute_relevance_total(ground_truth, search_function):
    return [compute_relevance(q, search_function) for q in tqdm(ground_truth)]

relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total[:5]

  0%|          | 0/515 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

## Hit Rate (a.k.a. Recall@k)

Fraction of queries where the correct document appears **anywhere** in the top-k.
Did search find it at all?

In [6]:
def hit_rate(relevance):
    cnt = sum(1 for line in relevance if 1 in line)
    return cnt / len(relevance)

hit_rate(relevance_total)

0.7572815533980582

## Mean Reciprocal Rank (MRR)

Like hit rate, but also rewards **position**. A hit at rank 1 scores 1.0, rank 2
scores 0.5, rank 3 scores 0.333, not-found scores 0. Averaged over queries.

> Hit Rate is the *ceiling* for MRR. MRR <= Hit Rate always. A high Hit Rate with
> a low MRR means the doc is retrieved but buried.

In [7]:
def mrr(relevance):
    total = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total += 1 / (rank + 1)  # +1 because python is 0-indexed
                break
    return total / len(relevance)

mrr(relevance_total)

0.599126213592233

In [8]:
# One reusable evaluator: takes a search function, returns both metrics.
def evaluate(ground_truth, search_function):
    rel = compute_relevance_total(ground_truth, search_function)
    return {"hit_rate": hit_rate(rel), "mrr": mrr(rel)}

evaluate(ground_truth, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.7572815533980582, 'mrr': 0.599126213592233}

## Use the metrics to tune search

Now that quality is a number, we can **sweep parameters** and pick the best. Try a
different boost:

In [9]:
def text_search_v2(query):
    return index.search(query, num_results=5, boost_dict={"question": 2.0, "section": 0.5})

evaluate(ground_truth, text_search_v2)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.7902912621359224, 'mrr': 0.6281553398058249}

In [10]:
# Sweep the question boost alone.
def search_boost(query, question_boost):
    return index.search(query, num_results=5, boost_dict={"question": question_boost, "section": 0.5})

for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    res = evaluate(ground_truth, lambda query, b=boost: search_boost(query, b))
    print(f"boost={boost:>4}: {res}")

  0%|          | 0/515 [00:00<?, ?it/s]

boost= 0.5: {'hit_rate': 0.8116504854368932, 'mrr': 0.6718770226537214}


  0%|          | 0/515 [00:00<?, ?it/s]

boost= 1.0: {'hit_rate': 0.8077669902912621, 'mrr': 0.658899676375404}


  0%|          | 0/515 [00:00<?, ?it/s]

boost= 3.0: {'hit_rate': 0.7572815533980582, 'mrr': 0.599126213592233}


  0%|          | 0/515 [00:00<?, ?it/s]

boost= 5.0: {'hit_rate': 0.7378640776699029, 'mrr': 0.5753721682847897}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7165048543689321, 'mrr': 0.5532038834951458}


In [11]:
# Full 3-parameter grid: question x answer x section boosts.
def search_boosts(query, question_boost, answer_boost, section_boost):
    return index.search(query, num_results=5, boost_dict={
        "question": question_boost, "section": section_boost, "answer": answer_boost})

results = []
for qb in [1.0, 2.0, 5.0]:
    for ab in [1.0, 2.0, 4.0, 10.0]:
        for sb in [0.1, 0.2, 0.5]:
            res = evaluate(ground_truth, lambda query, qb=qb, ab=ab, sb=sb:
                           search_boosts(query, qb, ab, sb))
            results.append({"question": qb, "answer": ab, "section": sb, **res})

df_results = pd.DataFrame(results).sort_values("mrr", ascending=False)
df_results.head(10)

  0%|          | 0/515 [00:00<?, ?it/s]

  0%|          | 0/515 [00:00<?, ?it/s]

  0%|          | 0/515 [00:00<?, ?it/s]

  0%|          | 0/515 [00:00<?, ?it/s]

  0%|          | 0/515 [00:00<?, ?it/s]

...

  0%|          | 0/515 [00:00<?, ?it/s]

  0%|          | 0/515 [00:00<?, ?it/s]

  0%|          | 0/515 [00:00<?, ?it/s]

  0%|          | 0/515 [00:00<?, ?it/s]

,question,answer,section,hit_rate,mrr
4,1.0,2.0,0.2,0.877670,0.723851
3,1.0,2.0,0.1,0.879612,0.722395
19,2.0,4.0,0.2,0.879612,0.722395
35,5.0,10.0,0.5,0.879612,0.722395
34,5.0,10.0,0.2,0.869903,0.720227
18,2.0,4.0,0.1,0.871845,0.720129
20,2.0,4.0,0.5,0.875728,0.720129
33,5.0,10.0,0.1,0.867961,0.719094
7,1.0,4.0,0.2,0.875728,0.716311
6,1.0,4.0,0.1,0.867961,0.714239


**Interpretation:** the best configs reach ~0.97 hit rate / ~0.88 MRR. Synthetic
questions tend to inflate these numbers (they're close to the FAQ wording), so
treat >95% with caution. The point of the exercise is the *method* (sweep ->
measure -> pick), which generalizes to vector search and hybrid setups.